# Fase 3 — pipeline de geração de questões de múltipla escolha

Geração iterativa de MCQ do domínio de FPSO, ancorada no corpus **Petrolês**,
com julgamento automático, controle de vícios de construção e critério de
parada por entropia.

## O que herda das fases anteriores

| da fase 2 | vira, na fase 3 |
|---|---|
| 30 pares de questões avaliados por especialistas | **banco seed** de few-shot (as vencedoras) |
| critérios do formulário (clareza, alternativas, correção técnica, relevância) | **rubrica do judge** — a nota deixa de ser arbitrária |
| prompt `PORTUGUESE_V15` | base do prompt do gerador |

## O pipeline (passos 0 a 9)

```
0. insumos     corpus Petrolês · 17 tópicos / 40 subtópicos com instrução do
               especialista · banco seed de questões de alta qualidade
1. recuperação query do subtópico -> N trechos do corpus
2. consolidação LLM resume os N trechos em um documento de geração enxuto
3. geração     lote de questões a partir do documento + few-shot sorteado
4. judgment    LLM-as-judge em lote, baixo esforço: descarta incorretas,
               atribui nota [0,1] e dificuldade
5. vícios      scorer determinístico: similaridade, comprimento, distratores
6. tolerância  passou do limite -> refinamento -> re-scorer -> passa ou descarta
7. storage     embedding(questão+correta), origem, nota, dificuldade, vícios
8. realimenta  questões de alto score entram no pool de few-shot
9. parada      entropia de Shannon do pool do subtópico; 5 rodadas sem ganho
               -> próximo documento / subtópico
```

## Modelos

| papel | deployment |
|---|---|
| gerador, judge, refinador | `gpt-5-petrobras` |
| extrator de facetas, consolidador | `gpt-4o-mini-petrobras` |
| embeddings | `sentence-transformers` multilíngue, local |

## Onde está o código

O notebook orquestra; o motor está em três módulos ao lado dele:

- `utils_fase3.py` — varredura do corpus, embeddings, recuperação, scorer de
  vícios, entropia, repositório e estado;
- `prompts_fase3.py` — todos os prompts (extrator, consolidador, gerador,
  judge, refinador);
- `seed_fase3.py` — construção do banco seed a partir do formulário da fase 2.

## 1. Setup

In [ ]:
# pip install openai httpx sentence-transformers scikit-learn numpy pandas
import sys, json, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

AQUI = Path.cwd()                      # pipeline/fase_3
RAIZ = (AQUI / ".." / "..").resolve()  # raiz do repositório
sys.path.insert(0, str(AQUI))
sys.path.insert(0, str(RAIZ))

import utils_fase3 as U
import prompts_fase3 as P
import seed_fase3 as S
from topicos import TOPICOS
from azure_openai_backend import AzureOpenAIBackend

pd.set_option("display.max_colwidth", 90)
print(f"raiz: {RAIZ}")
print(f"tópicos: {len(TOPICOS)} · subtópicos: {sum(len(v) for v in TOPICOS.values())}")

In [ ]:
cfg = U.Config(
    raiz=RAIZ,
    corpus_dir=RAIZ / "dataset" / "corpus-SemProcessamento-publico-PetrolesHibrido",
    # corpus_ignorar=["NILC.txt"],   # ver §4: exclui um arquivo sem mexer na pasta
    out_dir=AQUI / "saida_fase3",
    # --- modelos ---------------------------------------------------------
    modelo_forte="gpt-5-petrobras",
    modelo_leve="gpt-4o-mini-petrobras",
    # --- piloto ----------------------------------------------------------
    # O limiar de entropia é calibrado no fim deste notebook, com os ganhos
    # observados. Começa baixo de propósito: no piloto queremos ver as rodadas
    # improdutivas acontecerem, não evitá-las.
    limiar_ganho_entropia=0.001,
    rodadas_estagnadas_max=5,
    min_questoes_para_entropia=12,
    n_questoes_por_lote=6,
    n_exemplos_fewshot=3,
)
print(cfg.resumo())

In [ ]:
# Um deployment por papel. O backend cacheia em disco por hash do pedido, então
# reexecutar o notebook não repaga as chamadas idênticas.
llm_leve = AzureOpenAIBackend(deployment=cfg.modelo_leve,
                              temperature=0.2, max_tokens=3000)
llm_gerador = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=12000,
                                 reasoning_effort=cfg.gerador_reasoning_effort)
llm_judge = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=8000,
                               reasoning_effort=cfg.judge_reasoning_effort)
# O refinador é o mesmo gpt-5 do gerador — reescrever questão é tarefa de autor.
llm_refinador = llm_gerador

llm_leve.doctor()

## 2. Passo 0 — insumos fixos

### 2.1 Banco seed: as vencedoras do formulário da fase 2

Em cada um dos 30 pares, os especialistas escolheram a melhor questão. A
`nota_humana` é a margem de vitória normalizada pelo número de avaliadores —
com 2 avaliadores, `1.0` é unânime e `0.5` é decidida por um voto. Empates
ficam de fora.

In [ ]:
DIR_FASE2 = RAIZ / "pipeline" / "fase_2" / "questionarios" / "respostas_questionario"

banco_seed = S.construir_banco_seed(DIR_FASE2, min_nota_humana=0.5)
S.salvar_banco_seed(banco_seed, cfg.out_dir / "banco_seed.jsonl")

q = banco_seed[0]
print(f"\n--- exemplo ({q['id']}, nota humana {q['nota']}, {q['condicao']}) ---")
print(q["stem"])
for i, a in enumerate(q["alternatives"]):
    print(f"  {'>' if i == q['correct_answer_index'] else ' '} {chr(65+i)}) {a[:110]}")

### 2.2 Tópicos, subtópicos e a instrução do especialista

O problema prático: as instruções de `topicos.py` vão de 350 a **6.400
caracteres** e enumeram muitos assuntos independentes numa frase só.

Usar esse parágrafo inteiro como query de busca não funciona — o embedding do
parágrafo é a média de dez assuntos, e recupera o genérico de todos em vez do
específico de qualquer um. E usá-lo inteiro no prompt de geração dá sempre o
mesmo alvo: as rodadas se repetem, que é exatamente o que o critério de parada
deveria estar medindo.

In [ ]:
tamanhos = [(t, s, len(instr)) for t, subs in TOPICOS.items()
            for s, instr in subs.items()]
df_top = pd.DataFrame(tamanhos, columns=["topico", "subtopico", "chars_instrucao"])
print(f"{len(df_top)} subtópicos · instrução de {df_top.chars_instrucao.min()} a "
      f"{df_top.chars_instrucao.max()} caracteres (mediana {df_top.chars_instrucao.median():.0f})")
df_top.sort_values("chars_instrucao", ascending=False).head(8)

### 2.3 Escopo do piloto

Rodar os 40 subtópicos antes de calibrar o limiar de entropia é caro e cego.
O piloto roda três subtópicos de perfis diferentes — um bem coberto pelo
corpus, um normativo, um de gestão — até a estagnação, e é dele que sai o
limiar.

A varredura do corpus (§4), essa sim, roda uma vez só e vale para tudo: basta
incluir todos os subtópicos na lista de facetas quando quiser expandir.

In [ ]:
SUBTOPICOS_PILOTO = [
    ("Produção e Processo (Medição Fiscal e Sistema de Alívio)",
     "Medição Fiscal de óleo, medição fiscal de gás e transferência de custódia "
     "em operações de offloading"),
    ("Segurança Operacional, SGSO, Sistemas de Segurança e Gestão de Riscos", "SGSO"),
    ("Manutenção, Inspeção e Integridade de Ativos", "Manutenção e Plano de Manutenção"),
]

for t, s in SUBTOPICOS_PILOTO:
    assert s in TOPICOS[t], f"subtópico ausente em topicos.py: {s}"
    print(f"{t}\n  -> {s} ({len(TOPICOS[t][s])} chars)")

# Para rodar tudo:
# SUBTOPICOS_PILOTO = [(t, s) for t, subs in TOPICOS.items() for s in subs]

## 3. Facetas — a técnica para as instruções longas

O extrator (`gpt-4o-mini`) quebra cada instrução em **facetas**: recortes
técnicos independentes daquilo que o especialista pediu. Cada faceta é usada
três vezes, e é isso que torna a técnica barata:

| a faceta fornece | usado em |
|---|---|
| `termos_fortes` / `termos_apoio` | busca lexical no corpus (§4) |
| `query` | rerank semântico (§5) |
| `foco` | a instrução do especialista no prompt de geração (§7) |

O ganho não é só de recuperação. Como cada rodada mira uma faceta diferente, a
diversidade entre rodadas vem do desenho, e não de pedir "faça diferente" ao
modelo. Nada é inventado fora da instrução — o extrator só a decompõe.

O resultado fica em cache (`saida_fase3/indices/facetas.jsonl`); apague o
arquivo para reextrair.

In [ ]:
facetas = U.extrair_facetas(llm_leve, TOPICOS, cfg,
                            subtopicos_alvo=SUBTOPICOS_PILOTO)
por_sub = U.agrupar_por_subtopico(facetas)
print(f"\n{len(facetas)} facetas em {len(por_sub)} subtópicos")

In [ ]:
f = facetas[0]
print(f"faceta: {f.titulo}\n")
print(f"foco (vai para o prompt de geração):\n  {f.foco}\n")
print(f"query (rerank semântico):\n  {f.query}\n")
print(f"termos fortes: {f.termos_fortes}")
print(f"termos de apoio: {f.termos_apoio}")

## 4. Passo 1a — varredura do corpus (estágio lexical)

O corpus vive em `dataset/corpus-SemProcessamento-publico-PetrolesHibrido/` e são
**três** arquivos. `cfg.corpus_dir` aponta para a pasta e a varredura lê todos os
`.txt` dela, em ordem alfabética, tratando-os como um corpus só — a célula
abaixo lista exatamente o que vai ser lido, que é o momento barato de perceber
um arquivo a mais ou a menos.

Os três não têm a mesma forma, e isso importa:

| arquivo | tamanho | linhas | palavras/linha | `<NUMBER>` |
|---|---|---|---|---|
| `corpusPublico(sem IBICT)` | 0,54 GB | ~3,4 M | ~25 | sim |
| `corpusPublicoIBICT` | 0,43 GB | ~2,6 M | ~26 | sim |
| `NILC.txt` | 3,71 GB | ~110,5 M | ~7 | não |

**Por isso a janela é medida em palavras, não em linhas.** Uma janela de 12
linhas daria ~300 palavras nos dois primeiros e ~85 no NILC — trechos
incomparáveis, e no NILC fragmentos curtos demais para gerar questão. A
varredura acumula linhas até `alvo_palavras_trecho` (250) e desliza mantendo
`sobreposicao_trecho` (50%), então todo trecho sai do mesmo tamanho venha de
onde vier. A janela nunca cruza a fronteira entre arquivos.

### Sobre o NILC

O NILC é **95% das linhas do corpus** e é texto geral do português (notícias,
literatura, legendas — a primeira página são falas de um programa infantil),
não óleo e gás. O *gate* lexical exige um termo forte da faceta para o trecho
entrar, o que o filtra bem; mas ele responde por ~7 dos ~8 minutos de varredura.

Não estou excluindo por conta própria: rode com os três e olhe a tabela
`candidatos por arquivo` que a varredura imprime no fim. Se o NILC aparecer com
uma fatia irrisória de candidatos — o que é o esperado —, `cfg.corpus_ignorar =
["NILC.txt"]` corta a reindexação para menos de um minuto nas próximas vezes.

### Os dois estágios

1. **lexical, streaming** (aqui) — uma passada pontuando **todas as facetas ao
   mesmo tempo**, guardando os top-K trechos de cada uma. O léxico vem dos
   termos que o extrator produziu, não de listas escritas à mão. Um *gate* exige
   ao menos um termo forte (ou dois de apoio), o que corta falso positivo de
   termo genérico.
2. **semântico** (§5) — rerank e MMR só sobre esses candidatos, porque embeddar
   milhões de trechos é inviável.

Custa ~8 min com os três arquivos e roda **uma vez** por conjunto de facetas: o
resultado é verificado por assinatura (que inclui a lista de arquivos) e
reaproveitado depois. Mudou a lista de arquivos ou o conjunto de facetas, a
assinatura muda e a varredura refaz.

> `limite_linhas=500_000` faz um teste de fumaça rápido. Confira os candidatos,
> apague `saida_fase3/indices/candidatos*` e rode de novo sem o limite.

In [ ]:
U.descrever_corpus(cfg)

In [ ]:
%%time
U.varrer_corpus(facetas, cfg)          # limite_linhas=500_000 para um teste rápido
candidatos = U.carregar_candidatos(cfg)

resumo = pd.DataFrame([
    {"faceta": f.titulo[:44], "subtopico": f.subtopico[:34],
     "candidatos": len(candidatos.get(f.id, [])),
     "lex_max": max([c["lex_score"] for c in candidatos.get(f.id, [])], default=0)}
    for f in facetas
])
resumo

In [ ]:
# Contribuição de cada arquivo do corpus — é esta tabela que decide se vale
# manter o NILC na varredura ou pô-lo em cfg.corpus_ignorar.
manifesto = json.loads(
    (cfg.out_dir / "indices" / "candidatos_manifesto.json").read_text(encoding="utf-8"))
print(f"varredura: {manifesto['n_linhas']:,} linhas em "
      f"{manifesto['segundos'] / 60:.1f} min\n")
pd.DataFrame([
    {"arquivo": a,
     "linhas_lidas": manifesto["linhas_por_arquivo"].get(a, 0),
     "trechos_avaliados": manifesto["trechos_avaliados_por_arquivo"].get(a, 0),
     "candidatos_finais": manifesto["candidatos_por_arquivo"].get(a, 0)}
    for a in manifesto["arquivos"]
]).assign(share_candidatos=lambda d: (d.candidatos_finais / max(d.candidatos_finais.sum(), 1)).map("{:.1%}".format))

In [ ]:
# Inspeção manual — a etapa mais barata de detectar recuperação ruim.
f = facetas[0]
for c in candidatos[f.id][:2]:
    print(f"[{c['chunk_id']}] lex={c['lex_score']:.0f} termos={c['termos_fortes']}")
    print(c["texto"][:600], "...\n")

## 5. Passo 1b — rerank semântico, MMR e a fila de documentos

O embedder é o `sentence-transformers` multilíngue rodando local — o mesmo
vetor serve ao rerank, ao MMR, ao scorer de vícios (§8) e à entropia (§10),
o que mantém as quatro medidas na mesma geometria.

O score final combina lexical e semântico em z-scores; o **MMR** seleciona
trechos relevantes *e* diversos entre si (senão o documento consolidado vira
seis paráfrases da mesma passagem). Os selecionados são fatiados em blocos de
`n_trechos_por_documento` — cada bloco é um **documento** no sentido do passo 1,
e é a unidade que o critério de parada percorre quando o subtópico satura.

A fila intercala as facetas (`f0d0, f1d0, f2d0, f0d1, ...`): o subtópico cobre
a largura da instrução antes de aprofundar em qualquer recorte.

In [ ]:
emb = U.Embedder(cfg)

planos = {}
for subtopico, fs in por_sub.items():
    print(f"\n{subtopico}")
    planos[subtopico] = U.plano_de_documentos(fs, candidatos, emb, cfg)
    print(f"    fila: {len(planos[subtopico])} documentos")

## 6. Passo 2 — consolidação

O `gpt-4o-mini` transforma os N trechos brutos num documento técnico enxuto.
A regra dura do prompt é **não acrescentar conhecimento**: o consolidador
limpa, funde e descarta ruído, mas o que ele não recebeu não pode aparecer. Se
os trechos forem pobres, ele deve dizê-lo com o prefixo `AVISO:` — o que
transforma recuperação ruim em algo visível, em vez de virar questão inventada
lá na frente.

Documentos consolidados ficam em `saida_fase3/documentos/`.

In [ ]:
subtopico_demo = SUBTOPICOS_PILOTO[0][1]
plano_demo = planos[subtopico_demo]
fac_por_id = {f.id: f for f in facetas}

doc = plano_demo[0]
faceta = fac_por_id[doc["faceta_id"]]
documento = U.consolidar_documento(llm_leve, doc, faceta, cfg)

print(f"documento {doc['doc_id']} · faceta '{faceta.titulo}' · "
      f"{len(doc['trechos'])} trechos -> {len(documento.split())} palavras\n")
print(documento[:1500])

## 7. Passo 3 — geração

O prompt do gerador é o `PORTUGUESE_V15` da fase 2 com três acréscimos: o
`foco` da faceta (a instrução do especialista para aquele recorte), o documento
consolidado como **única fonte factual**, e o few-shot sorteado do banco de
alto score.

O few-shot é sorteado de **qualquer tópico**, de propósito: ele calibra forma —
profundidade, construção de distrator, tom — e não conteúdo. Exemplos do mesmo
subtópico convidariam o modelo a copiar o assunto.

O escopo do gerador é estreito de propósito: **enunciado, alternativas e
gabarito, e nada mais**. Dificuldade é do judge, que tem a rubrica na mão;
justificativa da correta não é produzida por ninguém; e a posição do gabarito é
sorteada no pós-processamento (§11). Cada um desses pedidos, quando está no
prompt, consome atenção do modelo em contabilidade que código faz melhor — e a
variação de posição entre A/B/C/D é o exemplo mais claro: é uma propriedade do
lote inteiro, que o modelo estima mal.

Um detalhe que evita um bug caro: o backend cacheia por hash do pedido, então
duas rodadas com o mesmo documento e o mesmo few-shot devolveriam questões
idênticas do cache. O prompt carrega um marcador de rodada — rodadas diferentes
divergem, a *mesma* rodada continua cacheável e o notebook pode ser reexecutado
de graça.

In [ ]:
repo = U.Repositorio(cfg, emb)
pool = U.PoolFewShot(banco_seed, repo, cfg)
print("pool de few-shot:", pool.composicao())

exemplos = pool.amostrar(excluir_subtopico=faceta.subtopico)
print(f"exemplos sorteados: {[e['id'] for e in exemplos]}")

geradas = U.gerar_lote(llm_gerador, doc, faceta, documento, exemplos,
                       rodada=1, cfg=cfg)
print(f"\n{len(geradas)} questões geradas\n")

q = geradas[0]
print(q["stem"])
for i, a in enumerate(q["alternatives"]):
    print(f"  {'>' if i == q['correct_answer_index'] else ' '} {chr(65+i)}) {a}")

## 8. Passo 4 — judge

Os quatro critérios do judge são deliberadamente os mesmos que os especialistas
usaram para justificar a escolha no formulário da fase 2, com pesos declarados
no prompt para o modelo não inventar a própria escala:

| critério | peso |
|---|---|
| correção técnica | 0,40 |
| clareza do enunciado | 0,20 |
| qualidade das alternativas | 0,25 |
| relevância para a operação de FPSOs | 0,15 |

A separação que importa: **descarte** é só para erro técnico — gabarito errado,
duas alternativas defensáveis, erro no enunciado. Redação ruim e distrator
fraco não são descarte, são nota baixa; se fossem descarte, o refinamento do
passo 6 nunca teria o que consertar.

Roda em lote e com `reasoning_effort` baixo, como pede o passo 4.

In [ ]:
aprovadas = U.julgar_lote(llm_judge, geradas, faceta.subtopico, cfg)
print(f"{len(geradas)} geradas -> {len(aprovadas)} passaram "
      f"(descartadas por corretude ou nota < {cfg.nota_minima_aprovacao})\n")

pd.DataFrame([{"id": q["id"], "nota": q["nota"], "dificuldade": q["difficulty"],
               **{k: q["judge"].get(k) for k in
                  ("correcao", "clareza", "alternativas", "relevancia")},
               "comentario": q["judge"].get("comentario", "")[:70]}
              for q in aprovadas])

## 9. Passo 5 — scorer de vícios

Três vícios de construção que deixam a questão respondível **sem saber o
conteúdo**. Nenhum precisa de LLM — todos são medidos:

- **similaridade** — a correta se parece mais com o enunciado do que os
  distratores (cosseno de embedding e sobreposição de palavras de conteúdo).
  Mede-se a *vantagem* da correta: distrator parecido com o enunciado não é
  vício, é distrator bom.
- **comprimento** — a correta destoa em tamanho. Em módulo: mais longa (o caso
  comum, o autor detalha a certa) e mais curta são ambos pista.
- **distratores** — distrator "lixo": sem relação semântica com o enunciado
  (elimina-se de bate-pronto) ou carregado de linguagem absolutista que a
  correta não tem (*apenas*, *exclusivamente*, *nunca*, *sempre*...).

Cada um é normalizado para `[0,1]` — 0 sem vício, 1 saturado — para que a
tolerância seja comparável entre eles.

### 9.1 Situando as tolerâncias contra o banco seed

Aqui a interpretação **não** é a mesma do limiar de entropia, e vale a pena não
confundir as duas.

As 22 questões do seed foram julgadas melhores por especialistas — mas o
formulário perguntava qual estava "melhor elaborada", justificando por clareza,
correção, alternativas e relevância. Vício de construção não estava na pauta, e
passou: em 95% das questões da fase 2 a correta é mais longa que a média dos
distratores.

Então o seed **não é o alvo, é o retrato do estado atual**. Adotar o p90 do seed
como tolerância institucionaliza o viés da fase 2 e o scorer deixa de reprovar
qualquer coisa. O uso correto é como referência de *custo*: para cada tolerância
candidata, quantas questões iriam para o refinador. Apertada demais, todo lote
vai para refinamento e o refinador vira o verdadeiro gerador; frouxa demais, o
passo 6 não faz nada.

In [ ]:
sugerido = U.calibrar_tolerancias(banco_seed, emb, cfg, percentil=75)

# Descomente para adotar as tolerâncias calibradas:
# cfg.tol_similaridade = sugerido["similaridade"]
# cfg.tol_comprimento  = sugerido["comprimento"]
# cfg.tol_distratores  = sugerido["distratores"]

In [ ]:
diagnosticos = {q["id"]: U.pontuar_vicios(q, emb, cfg) for q in aprovadas}

df_v = pd.DataFrame([{"id": qid, **U.resumo_vicios(d),
                      "reprovou": U.tem_vicio(d)}
                     for qid, d in diagnosticos.items()])
display(df_v)

for qid, d in diagnosticos.items():
    if U.tem_vicio(d):
        print(f"\n{qid}:")
        for k, v in d.items():
            if v["excedeu"]:
                print(f"  {k}: {v['valor']:.2f} > {v['limite']:.2f} — {v['detalhe']}")
        break

## 10. Passo 6 — filtro de tolerância e refinamento

Questão com vício vai para o refinador (`gpt-5`) com o **diagnóstico
quantitativo em mãos** — qual vício, quanto mediu, qual o limite. Sem isso o
modelo reescreve o que estava bom. Volta ao scorer; se reincidir, é descartada.
Uma tentativa, não um laço: se o refinamento não resolveu, o problema é da
questão.

In [ ]:
viciadas = [q for q in aprovadas if U.tem_vicio(diagnosticos[q["id"]])]
print(f"{len(viciadas)} de {len(aprovadas)} questões com vício acima da tolerância")

if viciadas:
    alvo = viciadas[0]
    refinada = U.refinar_questao(llm_refinador, alvo, diagnosticos[alvo["id"]],
                                 documento, cfg)
    if refinada:
        depois = U.pontuar_vicios(refinada, emb, cfg)
        print(f"\nantes:  {U.resumo_vicios(diagnosticos[alvo['id']])}")
        print(f"depois: {U.resumo_vicios(depois)}  -> "
              f"{'ainda com vício, descarta' if U.tem_vicio(depois) else 'aprovada'}")
        print(f"\nmudanças: {refinada['refinamento']}")
        print(f"\n{refinada['stem']}")
        for i, a in enumerate(refinada["alternatives"]):
            print(f"  {'>' if i == refinada['correct_answer_index'] else ' '} "
                  f"{chr(65+i)}) {a}")

## 11. Passos 7 e 8 — armazenamento e realimentação

Antes de armazenar, a posição do gabarito é **sorteada** — o passo que saiu do
prompt do gerador. Vem depois do refinamento, e como os vícios são medidos entre
correta e distratores (indiferentes à posição), nada do que veio antes é
invalidado. Controlado por `cfg.embaralhar_alternativas`: desligue se esse
embaralhamento já existir no seu pós-processamento.

A questão aprovada entra no repositório com o embedding do par
**questão + alternativa correta**, os `chunk_ids` de origem, a nota, a
dificuldade e os três scores de vício.

A realimentação não tem chave liga/desliga: o peso do seed decai conforme
questões próprias de alto score se acumulam, e a dependência do seed some
sozinha.

In [ ]:
if len(repo):
    print("posição do gabarito no repositório:",
          dict(Counter(chr(65 + q["correct_answer_index"]) for q in repo.questoes)))

print("\npeso do seed no sorteio conforme o banco próprio cresce:")
for n in (0, 5, 10, 20, 30, 40):
    print(f"  {n:>3} aprovadas de alto score -> peso do seed "
          f"{pool.peso_do_seed(n):.2f}")
print(f"\nagora: {pool.composicao()}")

## 12. Passo 9 — entropia e critério de parada

A pergunta é "este subtópico ainda rende questão nova?". A resposta é a
**entropia de Shannon normalizada** da distribuição dos embeddings
(questão + alternativa correta) acumulados no pool do subtópico, sobre uma
clusterização **fixa, treinada uma vez**. Se o codebook mudasse a cada rodada,
comparar rodadas não significaria nada.

**Por que o codebook é por subtópico, e não global do domínio:** com um
codebook global, as questões de um subtópico caem quase todas no mesmo
cluster — a entropia fica presa em zero e a parada dispara na primeira rodada.
O que interessa medir é a cobertura *dentro* do subtópico, então a
clusterização precisa resolver a estrutura fina daquele material. Ela é
treinada sobre os trechos candidatos do próprio subtópico. A comparação é
sempre entre rodadas do mesmo subtópico, então codebooks distintos entre
subtópicos não atrapalham.

Duas guardas contra falso positivo de estagnação: a primeira rodada não tem com
o que comparar, e um pool pequeno tem entropia baixa por falta de amostra
(`min_questoes_para_entropia`), não por saturação.

**Estado por subtópico** — só o que o passo 9 pede: o pool acumulado, o
histórico de entropia e o contador de rodadas estagnadas. Gravado a cada
rodada, então a execução é retomável.

In [ ]:
codebooks = {}
for subtopico, fs in por_sub.items():
    codebooks[subtopico] = U.codebook_do_subtopico(subtopico, fs, candidatos,
                                                   emb, cfg)

## 13. O laço completo do piloto (passos 3–9)

`executar_subtopico` encadeia geração → judge → vícios → refinamento →
armazenamento → entropia, avança de documento quando o subtópico satura e
termina quando a fila de documentos acaba.

É retomável: o estado é gravado a cada rodada e o cache do backend evita
repagar chamadas idênticas, então reexecutar continua de onde parou.

> Esta é a célula cara. Com 3 subtópicos × ~10 documentos × até 30 rodadas, o
> teto é alto — comece com `max_rodadas` baixo (5 ou 10) para ver o
> comportamento e o custo antes de soltar.

In [ ]:
%%time
estados = {}
for topico, subtopico in SUBTOPICOS_PILOTO:
    estados[subtopico] = U.executar_subtopico(
        subtopico=subtopico, topico=topico,
        facetas=por_sub[subtopico], plano=planos[subtopico],
        llm_leve=llm_leve, llm_forte=llm_gerador, llm_judge=llm_judge,
        pool=pool, repo=repo, emb=emb, codebook=codebooks[subtopico],
        cfg=cfg,
        max_rodadas=10,      # suba depois de ver o custo da primeira execução
    )

repo.salvar()
print(f"\nrepositório: {len(repo)} questões · pool: {pool.composicao()}")

## 14. Calibração do limiar de entropia

Agora sim, com dados. Cada histórico é partido em duas fases — **crescimento**,
até a entropia alcançar 90% do máximo que atingiu, e **platô**, dali em diante.
O limiar sai do percentil dos ganhos do platô: é o teto do ruído de uma rodada
que já não acrescenta cobertura. Fica travado em zero por baixo, porque um
limiar negativo exigiria que a entropia *caísse* para declarar estagnação — e o
critério nunca dispararia.

Se o aviso de "só viu crescimento" aparecer, nenhum subtópico saturou de fato:
suba `max_rodadas` e recalibre, senão o limiar sai de uma amostra que não
contém a informação que ele deveria resumir.

In [ ]:
historicos = {s: e.historico_entropia for s, e in estados.items()}
limiar = U.sugerir_limiar_entropia(historicos, percentil=75)

print("\nhistórico por subtópico:")
for s, h in historicos.items():
    print(f"  {s[:44]:<44} {len(h):>2} rodadas · "
          f"H de {h[0]:.3f} a {h[-1]:.3f}" if h else f"  {s}: sem rodadas")

# Adotando o limiar calibrado para a execução completa:
# cfg.limiar_ganho_entropia = limiar

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for s, h in historicos.items():
    if h:
        ax1.plot(range(1, len(h) + 1), h, marker="o", ms=3, label=s[:26])
        ax2.plot(range(2, len(h) + 1), np.diff(h), marker="o", ms=3, label=s[:26])
ax1.set(xlabel="rodada", ylabel="entropia normalizada",
        title="Cobertura acumulada do subtópico")
ax2.axhline(limiar, color="crimson", ls="--", lw=1,
            label=f"limiar sugerido {limiar:.4f}")
ax2.axhline(0, color="0.7", lw=0.8)
ax2.set(xlabel="rodada", ylabel="ganho de entropia",
        title="Ganho por rodada — é isto que o critério de parada lê")
ax1.legend(fontsize=8); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 15. Resultado

In [ ]:
df = repo.dataframe()
print(f"{len(df)} questões no repositório\n")

if len(df):
    print(df.groupby("subtopico").agg(
        questoes=("id", "count"), nota_media=("nota", "mean"),
        refinadas=("refinada", "sum")).round(3).to_string())
    print()
    print("dificuldade:", dict(df.dificuldade.value_counts()))
    print("vícios (média):", df[[c for c in df if c.startswith("vicio_")]]
          .mean().round(3).to_dict())
    print(f"alto score (>= {cfg.nota_minima_fewshot}): "
          f"{(df.nota >= cfg.nota_minima_fewshot).sum()}")
df.sort_values("nota", ascending=False).head(10)

In [ ]:
# Posição do gabarito — deve estar uniforme; concentração aqui significa que o
# embaralhamento do §11 não rodou (`cfg.embaralhar_alternativas`).
if len(repo):
    print("posição da correta:",
          dict(Counter(chr(65 + q["correct_answer_index"]) for q in repo.questoes)))

In [ ]:
# Export final.
destino = cfg.out_dir / "questoes_fase3.jsonl"
with open(destino, "w", encoding="utf-8") as fh:
    for q in repo.questoes:
        fh.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(repo)} questões -> {destino}")
print(f"embeddings         -> {repo.caminho_emb}")
print(f"estado por subtópico -> {cfg.out_dir / 'estado'}")
print(f"log de rodadas       -> {cfg.out_dir / 'logs' / 'rodadas.jsonl'}")
print(f"\nuso de tokens:")
for nome, llm in [("leve", llm_leve), ("gerador", llm_gerador), ("judge", llm_judge)]:
    u = llm.usage
    print(f"  {nome:<8} {u.calls:>4} chamadas ({u.cached_calls} de cache) · "
          f"{u.prompt_tokens:>9,} in · {u.completion_tokens:>8,} out")

## 16. Da calibração para a execução completa

Com o limiar calibrado, expandir é trocar duas linhas:

```python
SUBTOPICOS_PILOTO = [(t, s) for t, subs in TOPICOS.items() for s in subs]
cfg.limiar_ganho_entropia = limiar
```

e reexecutar de §3 em diante. O que muda de custo:

- **§3 facetas** — uma chamada leve por subtópico novo, com cache;
- **§4 varredura** — a assinatura muda com o conjunto de facetas, então há uma
  nova passada no corpus (~8 min com os três arquivos, ~1 min sem o NILC). É o
  único custo grande que reaparece; rode uma vez, já com todos os subtópicos, se
  pretende expandir logo;
- **§5 em diante** — proporcional ao número de subtópicos.

O que revisar antes de soltar nos 40:

1. os documentos consolidados que começam com `AVISO:` — recuperação fraca
   naquela faceta, e questão gerada dali não presta;
2. a taxa de descarte do judge por subtópico — descarte alto costuma ser
   documento ruim, não gerador ruim;
3. as tolerâncias de vício contra a calibração do §9.1;
4. a distribuição de dificuldade — se o judge estiver marcando quase tudo como
   "media", a escala não está discriminando.